# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AyeshaShahid-creator/Internship-flyrank-ML/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [2]:
# Setup — skip if already done earlier in this notebook
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MID_MONTH = "2026-03"
PRIOR_MONTH = "2026-02"
FACT = f"{BASE}/fact_content_daily_performance/**/*.parquet"

# Schema check — read the printed columns, then fix the constants below
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{FACT}') LIMIT 1").df())

# TODO: set these to the REAL column names from the output above
AVAILABILITY_COL = "gsc_data_available"
POSITION_COL = "gsc_avg_position"
CLICKS_COL = "gsc_clicks"
IMPRESSIONS_COL = "gsc_impressions"
CLIENT_COL = "client_hash_id"
CONTENT_COL = "content_hash_id"
print(con.sql(f"SELECT DISTINCT month FROM read_parquet('{FACT}') ORDER BY month").df())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      month
0   2025-01
1   2025-02
2   2025-03
3   2025-04
4   2025-05
5   2025-06
6   2025-07
7   2025-08
8   2025-09
9   2025-10
10  2025-11
11  2025-12
12  2026-01
13  2026-02
14  2026-03
15  2026-04
16  2026-05
17  2026-06


In [4]:
# Sealed test month setup — same 1-month gap pattern as dev (Feb→Mar)
TEST_PRIOR_MONTH = "2026-05"
TEST_CURRENT_MONTH = "2026-06"

# Features on the test "current" month (June) — same query as your dev features, different month
features_test = con.sql(f"""
    SELECT
        {CLIENT_COL} AS client_id, {CONTENT_COL} AS content_id,
        AVG({IMPRESSIONS_COL})                        AS avg_daily_impressions,
        SUM({CLICKS_COL}) * 1.0 / NULLIF(SUM({IMPRESSIONS_COL}), 0) AS avg_ctr,
        AVG({POSITION_COL})                           AS avg_position,
        STDDEV({IMPRESSIONS_COL})                      AS impression_volatility,
        COUNT(*)                                       AS days_with_data
    FROM read_parquet('{FACT}')
    WHERE month = '{TEST_CURRENT_MONTH}' AND {AVAILABILITY_COL} IS TRUE
    GROUP BY {CLIENT_COL}, {CONTENT_COL}
""").df()

# Trend/label for test — May vs June clicks, same logic as your Feb vs March label
prior_test = con.sql(f"""
    SELECT {CLIENT_COL} AS client_id, {CONTENT_COL} AS content_id,
           AVG({CLICKS_COL}) AS may_avg_clicks
    FROM read_parquet('{FACT}')
    WHERE month = '{TEST_PRIOR_MONTH}' AND {AVAILABILITY_COL} IS TRUE
    GROUP BY {CLIENT_COL}, {CONTENT_COL}
""").df()

current_test = con.sql(f"""
    SELECT {CLIENT_COL} AS client_id, {CONTENT_COL} AS content_id,
           AVG({CLICKS_COL}) AS jun_avg_clicks
    FROM read_parquet('{FACT}')
    WHERE month = '{TEST_CURRENT_MONTH}' AND {AVAILABILITY_COL} IS TRUE
    GROUP BY {CLIENT_COL}, {CONTENT_COL}
""").df()

trend_test = prior_test.merge(current_test, on=['client_id', 'content_id'])
trend_test['is_declining'] = (trend_test['jun_avg_clicks'] < trend_test['may_avg_clicks']).astype(int)

# Combine into final test set — same structure as your dev df
df_test = features_test.merge(
    trend_test[['client_id', 'content_id', 'is_declining']],
    on=['client_id', 'content_id']
).dropna()

# Define honest_cols based on the features in features_test
honest_cols = [
    'avg_daily_impressions',
    'avg_ctr',
    'avg_position',
    'impression_volatility',
    'days_with_data'
]
honest_cols = ['avg_daily_impressions', 'avg_ctr', 'avg_position', 'impression_volatility', 'days_with_data']

X_test = df_test[honest_cols]
y_test = df_test['is_declining']

print(df_test.shape)
df_test.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(168003, 8)


,client_id,content_id,avg_daily_impressions,avg_ctr,avg_position,impression_volatility,days_with_data,is_declining
1,client_3ffa76342f366962,content_6ee31d7d123af886,1.833333,0.090909,12.944444,0.752773,6,0
2,client_3ffa76342f366962,content_608a1d7a1ce91b44,2.481481,0.000000,14.632716,1.220667,27,1
3,client_3ffa76342f366962,content_65568e7c035d631f,1.307692,0.000000,33.358974,0.630425,13,1
4,client_3ffa76342f366962,content_50bf809645e7a321,1.000000,0.000000,6.800000,0.000000,5,1
5,client_3ffa76342f366962,content_478487452b5beac0,1.000000,0.250000,4.000000,0.000000,4,0


In [6]:
import pandas as pd
MID_MONTH = "2026-03"
PRIOR_MONTH = "2026-02"

features = con.sql(f"""
    SELECT
        {CLIENT_COL} AS client_id, {CONTENT_COL} AS content_id,
        AVG({IMPRESSIONS_COL})                        AS avg_daily_impressions,
        SUM({CLICKS_COL}) * 1.0 / NULLIF(SUM({IMPRESSIONS_COL}), 0) AS avg_ctr,
        AVG({POSITION_COL})                           AS avg_position,
        STDDEV({IMPRESSIONS_COL})                      AS impression_volatility,
        COUNT(*)                                       AS days_with_data
    FROM read_parquet('{FACT}')
    WHERE month = '{MID_MONTH}' AND {AVAILABILITY_COL} IS TRUE
    GROUP BY {CLIENT_COL}, {CONTENT_COL}
""").df()

prior = con.sql(f"""
    SELECT {CLIENT_COL} AS client_id, {CONTENT_COL} AS content_id,
           AVG({CLICKS_COL}) AS feb_avg_clicks
    FROM read_parquet('{FACT}')
    WHERE month = '{PRIOR_MONTH}' AND {AVAILABILITY_COL} IS TRUE
    GROUP BY {CLIENT_COL}, {CONTENT_COL}
""").df()

current = con.sql(f"""
    SELECT {CLIENT_COL} AS client_id, {CONTENT_COL} AS content_id,
           AVG({CLICKS_COL}) AS mar_avg_clicks
    FROM read_parquet('{FACT}')
    WHERE month = '{MID_MONTH}' AND {AVAILABILITY_COL} IS TRUE
    GROUP BY {CLIENT_COL}, {CONTENT_COL}
""").df()

trend = prior.merge(current, on=['client_id', 'content_id'])
trend['is_declining'] = (trend['mar_avg_clicks'] < trend['feb_avg_clicks']).astype(int)

df = features.merge(
    trend[['client_id', 'content_id', 'is_declining']],
    on=['client_id', 'content_id']
).dropna()

print(df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(129175, 8)


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

honest_cols = ['avg_daily_impressions', 'avg_ctr', 'avg_position', 'impression_volatility', 'days_with_data']

X = df[honest_cols]
y = df['is_declining']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

print("Model trained.")

Model trained.


In [10]:
from sklearn.metrics import precision_score

# Baseline: your Week 4 rule-based prediction on the test set
# (assumes action_label/action_score logic can be recomputed similarly on df_test —
#  for now, use "predict everyone as declining" as the simplest base rate baseline)
base_rate = y_test.mean()
import pandas as pd

# Model predictions on sealed June test set
y_pred_proba = clf.predict_proba(X_test)[:, 1]

# Precision@50 — top 50 highest-risk pages, what fraction actually declined
top_50_idx = pd.Series(y_pred_proba, index=X_test.index).nlargest(50).index
precision_at_50 = y_test.loc[top_50_idx].mean()

print(f"Base rate (share declining in June): {base_rate:.3f}")
print(f"Model Precision@50 on sealed June test: {precision_at_50:.3f}")


Base rate (share declining in June): 0.295
Model Precision@50 on sealed June test: 0.580


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
